# 02 — Feature Engineering

## Goal
Enrich the raw merged dataset with all engineered features before model training.
This notebook runs **once** and saves enriched datasets that all downstream notebooks consume.

## What this notebook does
1. Loads processed `train.parquet` + `test.parquet`
2. Concatenates train + test and sorts by `TransactionDT` — required for no-leakage aggregations
3. Computes **18 user aggregation features** (cumulative stats, email/device instability, velocity)
4. Computes **4 behavioral fingerprint features** (personal amount norms, time entropy)
5. Computes **2 product profile features** (new product flag, amount vs product typical)
6. Splits back into train/test using `isFraud.notna()` as the natural marker
7. Saves `train_enriched.parquet`, `test_enriched.parquet`, `y_train.parquet`

## Outputs
```
outputs/enriched/train_enriched.parquet  — enriched train features (no target)
outputs/enriched/test_enriched.parquet   — enriched test features
outputs/enriched/y_train.parquet         — target aligned by index to train_enriched
```

## No-leakage guarantee
All features use `expanding().shift(1)` or `rolling(closed='left')` — each row sees
only chronologically prior transactions. `isFraud` is never used in feature computation.

## Step 0 — Setup

In [1]:
import sys
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

# ---------------------------------------------------------------------------
# Path setup: notebook lives in v2/ — go one level up to reach project root.
# WHY os.getcwd() instead of __file__: __file__ is not available in notebooks.
# WHY abspath + join(.. ): guarantees correct absolute path regardless of how
# the notebook was launched (Jupyter, VS Code, Cursor).
# ---------------------------------------------------------------------------
PROJECT_ROOT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))

for subdir in ["", "src", "v0"]:
    p = os.path.join(PROJECT_ROOT_PATH, subdir)
    if p not in sys.path:
        sys.path.insert(0, p)

# PROJECT_ROOT is imported from config.py — single source of truth for all paths
from config import PROJECT_ROOT, TIME_COL

print(f"Project root : {PROJECT_ROOT}")
print(f"Resolved root: {PROJECT_ROOT_PATH}")
print(f"Paths added  : {[p for p in sys.path if PROJECT_ROOT_PATH in p]}")

Project root : C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection
Resolved root: c:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection
Paths added  : ['c:\\Users\\USER\\Desktop\\NOTEBOOKS\\HIT\\ML_Fraud_detection\\v0', 'c:\\Users\\USER\\Desktop\\NOTEBOOKS\\HIT\\ML_Fraud_detection\\src', 'c:\\Users\\USER\\Desktop\\NOTEBOOKS\\HIT\\ML_Fraud_detection\\']


## Step 1 — Load processed data

Load merged, memory-reduced `train.parquet` and `test.parquet`.
These files were created once by `data_loader.py` and contain the full feature set
from both the transaction and identity tables.

In [2]:
from data_loader import load_processed

# load_processed expects the /data directory containing train.parquet and test.parquet
train_raw, test_raw = load_processed(PROJECT_ROOT / "data")

# Verify basic shapes
print(f"Train shape : {train_raw.shape}")
print(f"Test shape  : {test_raw.shape}")
print(f"Train isFraud distribution: {train_raw['isFraud'].value_counts().to_dict()}")
print(f"Fraud rate  : {train_raw['isFraud'].mean():.4%}")

# isFraud exists ONLY in train — Kaggle test set never has target labels.
# We add a temporary 'is_train' marker before concat to split back correctly.
assert 'isFraud' in train_raw.columns, "isFraud missing from train!"
assert 'isFraud' not in test_raw.columns, "isFraud unexpectedly present in test!"
print("\nColumn check passed ✓ (isFraud in train only)")

>> Loading processed parquet files...
   Train: 590,540 rows × 439 cols (1565 MB)
   Test:  506,691 rows × 438 cols (1357 MB)
   Dtypes: {dtype('float32'): np.int64(399), dtype('O'): np.int64(32), dtype('int32'): np.int64(4), dtype('int64'): np.int64(2), dtype('int8'): np.int64(1), dtype('int16'): np.int64(1)}

   Train preview (top 7 rows):


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_36,id_37,id_38,DeviceType,DeviceInfo,tx_day,tx_dow,tx_hour,tx_dom,DeviceType_filled
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,None,None,None,None,None,1,1,0,2,No device info
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,None,None,None,None,None,1,1,0,2,No device info
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,None,None,None,None,None,1,1,0,2,No device info
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,None,None,None,None,None,1,1,0,2,No device info
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M,1,1,0,2,mobile
5,2987005,0,86510,49.0,W,5937,555.0,150.0,visa,226.0,...,None,None,None,None,None,1,1,0,2,No device info
6,2987006,0,86522,159.0,W,12308,360.0,150.0,visa,166.0,...,None,None,None,None,None,1,1,0,2,No device info


Train shape : (590540, 439)
Test shape  : (506691, 438)
Train isFraud distribution: {0: 569877, 1: 20663}
Fraud rate  : 3.4990%

Column check passed ✓ (isFraud in train only)


## Step 2 — Concatenate train + test, sort by time

All feature engineering must run on the **full dataset** (train + test concatenated).

**Why**: test transactions must see their prior history from train.
If we compute aggregations on train only, test rows have zero history — predictions become
unreliable and the pipeline misrepresents production behavior.

**No-leakage guarantee**: `isFraud` is present in the concatenated df as a split marker only.
It is NaN for all test rows and is **never used** in any feature computation.

In [3]:
# Add a binary marker BEFORE concat — this is the only reliable split key.
# WHY 'is_train' marker: isFraud exists only in train; after concat we need
# a column that unambiguously identifies train vs test rows.
train_raw = train_raw.copy()
test_raw  = test_raw.copy()
train_raw['is_train'] = 1
test_raw['is_train']  = 0

# Concatenate train and test into a single chronological dataframe.
# ignore_index=True gives clean 0-based integer index across both sets.
full = pd.concat([train_raw, test_raw], ignore_index=True)

# Sort by TransactionDT — mandatory before any expanding/rolling aggregation.
# WHY: groupby + expanding().shift(1) respects row order, not DT values.
# Without sorting, 'prior' rows may be chronologically later — that is leakage.
full = full.sort_values("TransactionDT").reset_index(drop=True)

# Sanity checks
n_train = (full['is_train'] == 1).sum()
n_test  = (full['is_train'] == 0).sum()
assert n_train == len(train_raw), f"Train count mismatch: {n_train} vs {len(train_raw)}"
assert n_test  == len(test_raw),  f"Test count mismatch: {n_test} vs {len(test_raw)}"
assert full['isFraud'].isna().sum() == n_test, "isFraud NaN count must equal test rows"

print(f"Full dataset shape   : {full.shape}")
print(f"Train rows (is_train=1): {n_train:,}")
print(f"Test  rows (is_train=0): {n_test:,}")
print(f"TransactionDT range  : {full['TransactionDT'].min()} → {full['TransactionDT'].max()}")

Full dataset shape   : (1097231, 478)
Train rows (is_train=1): 590,540
Test  rows (is_train=0): 506,691
TransactionDT range  : 86400 → 34214345


## Step 3 — User aggregation features (18 features)

Computes per-user (card1 + addr1) behavior statistics, strictly before each transaction:
- **Cumulative stats**: tx_count, amt_mean/std/min/max, tx_amt_ratio, time_since_last_tx, delta_amt
- **Email instability**: nunique_P/R_email, is_new_P/R_email, is_same_email_domain
- **Device instability**: nunique_device, is_new_device
- **Velocity**: tx_count_last_3d / 7d / 30d

In [4]:
from preproc_agg import compute_user_aggregations

# compute_user_aggregations returns (df_with_features, list_of_new_col_names).
# It internally re-sorts by TransactionDT and resets index — full remains aligned.
full, agg_feature_cols = compute_user_aggregations(full, verbose=True)

print(f"\nAggregation features added ({len(agg_feature_cols)}): {agg_feature_cols}")
print(f"Full shape after aggregations: {full.shape}")

FEATURE ENGINEERING — User Behavior Aggregations
   Group key:    ['card1', 'addr1']
   Shape before: (1097231, 478)

>> Feature Engineering: Cumulative Aggregations...
>> Feature Engineering: Email Instability...
>> Feature Engineering: Device Instability...
>> Feature Engineering: Rolling Window Velocity...

   Shape after:  (1097231, 496)
   New features (18):
     + tx_count
     + tx_amt_mean
     + tx_amt_std
     + tx_amt_min
     + tx_amt_max
     + tx_amt_ratio
     + time_since_last_tx
     + delta_amt
     + nunique_P_email
     + is_new_P_email
     + nunique_R_email
     + is_new_R_email
     + is_same_email_domain
     + nunique_device
     + is_new_device
     + tx_count_last_3d
     + tx_count_last_7d
     + tx_count_last_30d

Aggregation features added (18): ['tx_count', 'tx_amt_mean', 'tx_amt_std', 'tx_amt_min', 'tx_amt_max', 'tx_amt_ratio', 'time_since_last_tx', 'delta_amt', 'nunique_P_email', 'is_new_P_email', 'nunique_R_email', 'is_new_R_email', 'is_same_email_doma

## Step 4 — Behavioral fingerprint features (4 features)

Captures how much each transaction deviates from the **personal norm** of that user:
- `amt_vs_personal_median` — current amount / user's historical median amount
- `amt_z_score`           — (current amount − user mean) / user std
- `hour_vs_typical`       — deviation from user's typical transaction hour
- `uid_time_entropy`      — entropy of user's historical amount distribution (low = bot-like)

In [5]:
from preproc_behavioral import compute_behavioral_features

full, behavioral_feature_cols = compute_behavioral_features(full, verbose=True)

print(f"\nBehavioral features added ({len(behavioral_feature_cols)}): {behavioral_feature_cols}")
print(f"Full shape after behavioral features: {full.shape}")

FEATURE ENGINEERING — Behavioral Fingerprint
   Group key    : ['card1', 'addr1']
   Shape before : (1097231, 496)

   Computing amt_vs_personal_median ...
   Computing amt_z_score ...
   Computing hour_vs_typical ...
   Computing uid_time_entropy (entropy loop — may take ~1-2 min) ...

   Shape after  : (1097231, 500)
   New features : ['amt_vs_personal_median', 'amt_z_score', 'hour_vs_typical', 'uid_time_entropy']

   NaN check: 0 unexpected NaN in new features ✓

Behavioral features added (4): ['amt_vs_personal_median', 'amt_z_score', 'hour_vs_typical', 'uid_time_entropy']
Full shape after behavioral features: (1097231, 500)


## Step 5 — Product profile features (2 features)

Captures how each transaction relates to the **product-level norm**:
- `is_new_product`         — 1 if user is buying this ProductCD for the first time
- `amt_vs_product_typical` — current amount / median amount for this ProductCD

In [6]:
from preproc_product import compute_product_features

full, product_feature_cols = compute_product_features(full, verbose=True)

print(f"\nProduct features added ({len(product_feature_cols)}): {product_feature_cols}")
print(f"Full shape after product features: {full.shape}")

FEATURE ENGINEERING — Product Profile
   Group key    : ['card1', 'addr1']
   Product col  : ProductCD
   Shape before : (1097231, 500)

   Computing is_new_product ...

   Shape after  : (1097231, 501)
   is_new_product distribution: {0: 1024835, 1: 72396}
   First-time product purchases: 6.6% of all transactions
   NaN check: 0 unexpected NaN ✓

Product features added (1): ['is_new_product']
Full shape after product features: (1097231, 501)


## Step 6 — Split back into train / test + extract target

`isFraud.notna()` is the **natural split marker** — no artificial flags needed.
- Train rows: `isFraud` is 0 or 1
- Test rows:  `isFraud` is NaN (never provided in the Kaggle test set)

`y_train` is extracted **before** dropping `isFraud` and shares the same index as
`train_enriched` — alignment is verified by assert before saving.

In [7]:
# Split back using the is_train marker — set before concat, never modified.
train_enriched = full[full['is_train'] == 1].copy()
test_enriched  = full[full['is_train'] == 0].copy()

# Extract target BEFORE dropping any columns — index alignment is guaranteed here.
# y_train shares the same index as train_enriched.
y_train = train_enriched['isFraud'].astype(int)

# ── Alignment assertions ──────────────────────────────────────────────────────
assert len(train_enriched) == len(train_raw), (
    f"Train size mismatch: got {len(train_enriched)}, expected {len(train_raw)}"
)
assert len(test_enriched) == len(test_raw), (
    f"Test size mismatch: got {len(test_enriched)}, expected {len(test_raw)}"
)
assert y_train.index.equals(train_enriched.index), (
    "y_train index does not match train_enriched index — target misalignment!"
)
assert test_enriched['isFraud'].isna().all(), (
    "Test set contamination: isFraud values found in test rows!"
)
assert set(y_train.unique()).issubset({0, 1}), (
    f"Unexpected target values: {y_train.unique()}"
)

print("All alignment assertions passed ✓")
print(f"train_enriched shape : {train_enriched.shape}")
print(f"test_enriched  shape : {test_enriched.shape}")
print(f"y_train shape        : {y_train.shape}")
print(f"y_train distribution : {y_train.value_counts().to_dict()}")
print(f"Fraud rate           : {y_train.mean():.4%}")

All alignment assertions passed ✓
train_enriched shape : (590540, 501)
test_enriched  shape : (506691, 501)
y_train shape        : (590540,)
y_train distribution : {0: 569877, 1: 20663}
Fraud rate           : 3.4990%


## Step 7 — Drop isFraud from features + save

`isFraud` was needed as a split marker — now it must be removed from feature DataFrames
before saving. `y_train` is already extracted and saved separately.

**Index semantics**: `train_enriched` and `y_train` share the same integer index.
Any downstream operation using `.loc[]` on either will remain aligned.

In [8]:
# Build output directory path from PROJECT_ROOT (single source of truth)
enriched_dir = PROJECT_ROOT / "outputs" / "enriched"
enriched_dir.mkdir(parents=True, exist_ok=True)

# Drop isFraud AND is_train — both were markers, neither is a feature.
# y_train is already extracted and saved separately.
cols_to_drop_markers = ['isFraud', 'is_train']
train_enriched = train_enriched.drop(
    columns=[c for c in cols_to_drop_markers if c in train_enriched.columns]
)
test_enriched = test_enriched.drop(
    columns=[c for c in cols_to_drop_markers if c in test_enriched.columns]
)

# Save all three artifacts — index=True preserves row alignment between features and target
train_enriched.to_parquet(enriched_dir / "train_enriched.parquet", index=True)
test_enriched.to_parquet( enriched_dir / "test_enriched.parquet",  index=True)
y_train.to_frame().to_parquet(enriched_dir / "y_train.parquet",    index=True)

print("Saved:")
print(f"  {enriched_dir / 'train_enriched.parquet'}")
print(f"  {enriched_dir / 'test_enriched.parquet'}")
print(f"  {enriched_dir / 'y_train.parquet'}")
print(f"\ntrain_enriched final shape : {train_enriched.shape}")
print(f"test_enriched  final shape : {test_enriched.shape}")

Saved:
  C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\enriched\train_enriched.parquet
  C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\enriched\test_enriched.parquet
  C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\enriched\y_train.parquet

train_enriched final shape : (590540, 499)
test_enriched  final shape : (506691, 499)


## Step 8 — Feature engineering summary

Quick overview of all new features added in this notebook.

In [9]:
all_new_cols = agg_feature_cols + behavioral_feature_cols + product_feature_cols

print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)
print(f"  Raw features (v0 baseline)    : {len(train_raw.columns) - 3}")  # minus isFraud, TransactionID, TransactionDT
print(f"  Aggregation features (Step 3) : {len(agg_feature_cols)}")
print(f"  Behavioral features  (Step 4) : {len(behavioral_feature_cols)}")
print(f"  Product features     (Step 5) : {len(product_feature_cols)}")
print(f"  Total new features            : {len(all_new_cols)}")
print(f"  Total features in train_enriched (incl. ID/DT cols): {train_enriched.shape[1]}")
print()

# Check for NaN in new features — unexpected NaN outside of first-transaction fill
new_feat_nan = train_enriched[all_new_cols].isna().sum()
new_feat_nan = new_feat_nan[new_feat_nan > 0]
if len(new_feat_nan) == 0:
    print("  NaN check: no unexpected NaN in new features ✓")
else:
    print(f"  NaN in new features (first-transaction fill expected):\n{new_feat_nan}")

FEATURE ENGINEERING SUMMARY
  Raw features (v0 baseline)    : 437
  Aggregation features (Step 3) : 18
  Behavioral features  (Step 4) : 4
  Product features     (Step 5) : 1
  Total new features            : 23
  Total features in train_enriched (incl. ID/DT cols): 499

  NaN in new features (first-transaction fill expected):
tx_count    65706
dtype: int64
